### Functions with Configuration Support

In [ ]:
# ============================================================================
# IMPORTS - Put these at the TOP of your notebook
# ============================================================================
import sys
sys.path.append(r'D:\Neural-Pipeline\source')

import numpy as np
import gc
from pathlib import Path
from IPython.display import clear_output

# Import the class directly at the top
from analysis_pseudopopulation.DecoderCrossTester import DecoderCrossTester
from analysis_pseudopopulation import SlidingWindowDecoder

# Verify import worked
print(f"DecoderCrossTester type: {type(DecoderCrossTester)}")
print(f"Successfully imported DecoderCrossTester class")


# ============================================================================
# FUNCTIONS
# ============================================================================

def load_session_data(subject, date):
    """Load all data for a given subject and date"""
    from analysis_utils.NeuralDataLoader import NeuralDataLoader, Dots3DMPConfig
    
    loader = NeuralDataLoader()
    loader.load_session(subject, date)
    config = Dots3DMPConfig(subject)

    stimOn_spikes = loader.get_spike_data(alignment='stimOn', good_units_only=True, good_trials_only=True)
    saccOnset_spikes = loader.get_spike_data(alignment='saccOnset', good_units_only=True, good_trials_only=True)
    postTargHold_spikes = loader.get_spike_data(alignment='postTargHold', good_units_only=True, good_trials_only=True)
    tuning_spikes = loader.get_tuning_data(good_units_only=True, good_trials_only=True)

    behavior_dots3DMP = loader.get_behavioral_data(task='dots3DMP', good_trials_only=True)
    behavior_tuning = loader.get_behavioral_data(task='tuning', good_trials_only=True)
    behavior_converted = config.convert_behavioral_data(behavior_dots3DMP, task='dots3DMP')
    behavior_tuning_converted = config.convert_behavioral_data(behavior_tuning, task='tuning')

    unit_info = loader.get_unit_info(good_units_only=True)
    MST_units = loader.get_units_by_area(unit_info, area_name='MST')
    VPS_units = loader.get_units_by_area(unit_info, area_name='VPS')
    dual_units = loader.get_units_by_area(unit_info, area_name='dual')

    time_info = config.get_time_Info('dots3DMP')
    time_info_tuning = config.get_time_Info('tuning')
    time_axes_dots3DMP = config.get_time_axes('dots3DMP')

    spikes_data = {
        'stimOn': stimOn_spikes,
        'saccOnset': saccOnset_spikes,
        'postTargHold': postTargHold_spikes
    }
    
    units_data = {
        'MST': MST_units,
        'VPS': VPS_units,
        'dual': dual_units
    }
    
    return {
        'loader': loader,
        'config': config,
        'spikes_data': spikes_data,
        'behavior_converted': behavior_converted,
        'behavior_tuning_converted': behavior_tuning_converted,
        'unit_info': unit_info,
        'units_data': units_data,
        'time_axes_dots3DMP': time_axes_dots3DMP,
        'time_info': time_info,
        'time_info_tuning': time_info_tuning
    }


def parse_conditions(conditions_config):
    """Parse condition configuration into list of (mod, coh) tuples."""
    if conditions_config == 'all':
        conditions = []
        for mod in [1, 2, 3]:
            for coh in [1, 2]:
                if mod == 1 and coh == 2:
                    continue
                conditions.append((mod, coh))
        return conditions
    elif isinstance(conditions_config, list):
        return conditions_config
    elif isinstance(conditions_config, dict):
        modalities = conditions_config.get('modalities', [1, 2, 3])
        coherences = conditions_config.get('coherences', [1, 2])
        conditions = []
        for mod in modalities:
            for coh in coherences:
                if mod == 1 and coh == 2:
                    continue
                conditions.append((mod, coh))
        return conditions
    else:
        raise ValueError(f"Invalid conditions_config: {conditions_config}")


def parse_deltas(delta_config, behavior_data, train_delta=0):
    """Parse delta configuration into list of delta values to test."""
    
    # Check if delta exists in behavior data
    if 'delta' not in behavior_data:
        print(f"  ⚠ WARNING: 'delta' not found in behavior_data!")
        print(f"  Available keys: {list(behavior_data.keys())}")
        return [0]  # Return default
    
    unique_deltas = np.unique(behavior_data['delta'])
    available_deltas = [d for d in unique_deltas 
                       if d is not None and not (isinstance(d, float) and np.isnan(d))]
    available_deltas = sorted(available_deltas)
    
    print(f"  Available deltas in data: {available_deltas}")
    
    if delta_config == 'all':
        selected_deltas = available_deltas
        print(f"  Using ALL deltas: {selected_deltas}")
        return selected_deltas
    elif isinstance(delta_config, list):
        selected_deltas = [d for d in delta_config if d in available_deltas]
        missing = [d for d in delta_config if d not in available_deltas]
        if missing:
            print(f"  ⚠ Requested deltas not in data (skipped): {missing}")
        print(f"  Using deltas: {selected_deltas}")
        return selected_deltas
    elif isinstance(delta_config, dict) and 'exclude' in delta_config:
        exclude = delta_config['exclude']
        selected_deltas = [d for d in available_deltas if d not in exclude]
        print(f"  Excluding deltas: {exclude}")
        print(f"  Using deltas: {selected_deltas}")
        return selected_deltas
    else:
        raise ValueError(f"Invalid delta_config: {delta_config}")


def get_trained_file_path(subject, date, area, target, alignment, train_mod, train_coh, partial_regress=True):
    """Get path to trained model file (train==test condition)"""
    regress = 'partialregress' if partial_regress else 'noregress'
    base_dir = Path(rf'D:\Neural-Pipeline\results\analysis_pseudopopulation\decoders_{regress}')
    filename = f"{subject}_{date}_{area}_{target}_{alignment}_train_mod{train_mod}_coh{train_coh}_test_mod{train_mod}_coh{train_coh}_results.npy"
    return base_dir / filename


def get_delta_test_file_path(subject, date, area, target, alignment, train_mod, train_coh, partial_regress=True):
    """Get path to delta test file (separate file for all delta tests)."""
    regress = 'partialregress' if partial_regress else 'noregress'
    base_dir = Path(rf'D:\Neural-Pipeline\results\analysis_pseudopopulation\decoders_{regress}_deltatests')
    base_dir.mkdir(parents=True, exist_ok=True)
    filename = f"{subject}_{date}_{area}_{target}_{alignment}_trainMod{train_mod}Coh{train_coh}_deltatest.npy"
    return base_dir / filename


def get_crosstest_file_path(subject, date, area, target, alignment, train_mod, train_coh, 
                            test_mod, test_coh, partial_regress=True):
    """Get path to crosstest file (train!=test condition)"""
    regress = 'partialregress' if partial_regress else 'noregress'
    base_dir = Path(rf'D:\Neural-Pipeline\results\analysis_pseudopopulation\decoders_{regress}_crosstest')
    base_dir.mkdir(parents=True, exist_ok=True)
    filename = f"{subject}_{date}_{area}_{target}_{alignment}_trainMod{train_mod}Coh{train_coh}_testMod{test_mod}Coh{test_coh}_crosstest.npy"
    return base_dir / filename


def ensure_training_exists(subject, date, data_dict, area, target, train_mod, train_coh, 
                           valid_units, permutation_test=False, partial_regress=False):
    """Ensure training file exists. If not, run training."""
    alignments = ['stimOn', 'saccOnset', 'postTargHold']
    all_exist = True
    
    for alignment in alignments:
        filepath = get_trained_file_path(subject, date, area, target, alignment, 
                                        train_mod, train_coh, partial_regress)
        if not filepath.exists():
            all_exist = False
            break
    
    if all_exist:
        print(f"  ✓ Training already exists for {area}/{target}/mod{train_mod}coh{train_coh}")
        return True
    
    print(f"  → Training {area}/{target}/mod{train_mod}coh{train_coh}...")
    
    decoder = SlidingWindowDecoder(
        subject, date,
        run_permutation_test=permutation_test,
        partial_regress=partial_regress,
        n_permutations=20
    )
    
    try:
        results = decoder.run_decoding_analysis_cv(
            spikes_data=data_dict['spikes_data'],
            behavior_data=data_dict['behavior_converted'],
            time_axes=data_dict['time_axes_dots3DMP'],
            area=area,
            target=target,
            train_mod=train_mod,
            train_coh=train_coh,
            test_mod=train_mod,
            test_coh=train_coh,
            valid_units=valid_units,
            save_results=True
        )
        
        if results:
            results.clear()
            del results
        
        print(f"  ✓ Training completed for {area}/{target}/mod{train_mod}coh{train_coh}")
        return True
        
    except Exception as e:
        print(f"  ✗ Training failed: {e}")
        return False
    finally:
        del decoder
        gc.collect()


def test_delta_conditions(subject, date, data_dict, area, target, alignment,
                          train_mod, train_coh, valid_units, 
                          deltas_to_test, partial_regress=False):
    """Test on specified delta values and save as SEPARATE delta test file."""
    
    delta_filepath = get_delta_test_file_path(
        subject, date, area, target, alignment,
        train_mod, train_coh, partial_regress
    )
    
    if delta_filepath.exists():
        existing_results = np.load(delta_filepath, allow_pickle=True).item()
        existing_deltas = list(existing_results.get('delta_results', {}).keys())
        new_deltas = [d for d in deltas_to_test if f'delta_{d}' not in existing_deltas]
        
        if not new_deltas:
            print(f"    ✓ All delta tests already complete for {alignment}")
            return True
        
        print(f"    → Updating delta tests for {alignment}: {new_deltas}")
        delta_results = existing_results['delta_results'].copy()
    else:
        print(f"    → Testing deltas for {alignment}: {deltas_to_test}")
        delta_results = {}
        new_deltas = deltas_to_test
    
    trained_filepath = get_trained_file_path(
        subject, date, area, target, alignment,
        train_mod, train_coh, partial_regress
    )
    
    if not trained_filepath.exists():
        print(f"    ⚠ Training file not found: {trained_filepath}")
        return False
    
    trained_results = np.load(trained_filepath, allow_pickle=True).item()
    
    tester = DecoderCrossTester(subject, date, partial_regress=partial_regress)
    
    for delta_val in new_deltas:
        try:
            delta_result = tester.test_on_new_condition(
                trained_results=trained_results,
                spikes_data=data_dict['spikes_data'],
                behavior_data=data_dict['behavior_converted'],
                test_mod=train_mod,
                test_coh=train_coh,
                test_delta=delta_val,
                valid_units=valid_units,
                heading_filter=None,
                save_results=False
            )
            delta_results[f'delta_{delta_val}'] = delta_result
            print(f"        ✓ Delta {delta_val} tested")
        except Exception as e:
            print(f"        ✗ Delta {delta_val} failed: {e}")
    
    delta_test_results = {
        'delta_results': delta_results,
        'trained_from': {
            'area': area,
            'target': target,
            'alignment': alignment,
            'train_mod': train_mod,
            'train_coh': train_coh
        },
        'metadata': {
            'tested_deltas': list(delta_results.keys()),
            'partial_regress': partial_regress
        }
    }
    
    np.save(delta_filepath, delta_test_results, allow_pickle=True)
    print(f"    ✓ Delta tests saved to {delta_filepath.name}")
    
    del tester, trained_results
    return True


def test_cross_condition(subject, date, data_dict, area, target, alignment,
                        train_mod, train_coh, test_mod, test_coh, 
                        valid_units, deltas_to_test, partial_regress=False):
    """Test on different modality/coherence with specified deltas."""
    
    crosstest_filepath = get_crosstest_file_path(
        subject, date, area, target, alignment,
        train_mod, train_coh, test_mod, test_coh, partial_regress
    )
    
    if crosstest_filepath.exists():
        existing_results = np.load(crosstest_filepath, allow_pickle=True).item()
        existing_deltas = list(existing_results.get('test_results_by_delta', {}).keys())
        new_deltas = [d for d in deltas_to_test if f'delta_{d}' not in existing_deltas]
        
        if not new_deltas:
            print(f"    ✓ Crosstest already complete")
            return True
        
        print(f"    → Updating crosstest with new deltas: {new_deltas}")
        all_delta_results = existing_results['test_results_by_delta'].copy()
    else:
        print(f"    → Cross-testing on deltas: {deltas_to_test}")
        all_delta_results = {}
        new_deltas = deltas_to_test
    
    trained_filepath = get_trained_file_path(
        subject, date, area, target, alignment,
        train_mod, train_coh, partial_regress
    )
    
    if not trained_filepath.exists():
        print(f"    ✗ Training file not found")
        return False
    
    trained_results = np.load(trained_filepath, allow_pickle=True).item()
    
    tester = DecoderCrossTester(subject, date, partial_regress=partial_regress)
    
    for delta_val in new_deltas:
        try:
            delta_result = tester.test_on_new_condition(
                trained_results=trained_results,
                spikes_data=data_dict['spikes_data'],
                behavior_data=data_dict['behavior_converted'],
                test_mod=test_mod,
                test_coh=test_coh,
                test_delta=delta_val,
                valid_units=valid_units,
                heading_filter=None,
                save_results=False
            )
            all_delta_results[f'delta_{delta_val}'] = delta_result
            print(f"        ✓ Delta {delta_val}")
        except Exception as e:
            print(f"        ✗ Delta {delta_val} failed: {e}")
    
    crosstest_results = {
        'test_results_by_delta': all_delta_results,
        'trained_from': {
            'area': area,
            'target': target,
            'alignment': alignment,
            'train_mod': train_mod,
            'train_coh': train_coh
        },
        'test_condition': {
            'test_mod': test_mod,
            'test_coh': test_coh
        },
        'metadata': {
            'tested_deltas': list(all_delta_results.keys()),
            'partial_regress': partial_regress
        }
    }
    
    np.save(crosstest_filepath, crosstest_results, allow_pickle=True)
    print(f"    ✓ Saved crosstest to {crosstest_filepath.name}")
    
    del tester, trained_results
    return True


def run_full_decoding_pipeline(subject, date, data_dict, 
                               train_conditions, test_conditions, test_deltas,
                               areas, targets, permutation_test=False,
                               partial_regress=False):
    """Full pipeline with custom condition configuration."""
    alignments = ['stimOn', 'saccOnset', 'postTargHold']
    
    train_conds = parse_conditions(train_conditions)
    test_conds = parse_conditions(test_conditions)
    
    print(f"\nConfiguration:")
    print(f"  Areas: {areas}")
    print(f"  Targets: {targets}")
    print(f"  Train conditions: {train_conds}")
    print(f"  Test conditions: {test_conds}")
    print(f"\nParsing delta configuration...")
    deltas = parse_deltas(test_deltas, data_dict['behavior_converted'])
    print()
    
    for area in areas:
        print(f"\n{'='*80}")
        print(f"=== Processing area: {area} ===")
        print(f"{'='*80}")
        
        valid_units = data_dict['units_data'][area]
        
        for target in targets:
            print(f"\n{'*'*60}")
            print(f"*** Target: {target} ***")
            print(f"{'*'*60}")
            
            for train_mod, train_coh in train_conds:
                print(f"\n--- Training condition: mod={train_mod}, coh={train_coh} ---")
                
                training_success = ensure_training_exists(
                    subject, date, data_dict, area, target,
                    train_mod, train_coh, valid_units,
                    permutation_test, partial_regress
                )
                
                if not training_success:
                    print(f"  ✗ Training failed, skipping this condition")
                    continue
                
                print(f"\n  → Testing delta conditions (separate file)...")
                for alignment in alignments:
                    test_delta_conditions(
                        subject, date, data_dict, area, target, alignment,
                        train_mod, train_coh, valid_units, deltas, partial_regress
                    )
                
                cross_test_conds = [(m, c) for m, c in test_conds 
                                   if not (m == train_mod and c == train_coh)]
                
                if cross_test_conds:
                    print(f"\n  → Cross-condition testing...")
                    for test_mod, test_coh in cross_test_conds:
                        print(f"    Testing on mod={test_mod}, coh={test_coh}")
                        
                        for alignment in alignments:
                            test_cross_condition(
                                subject, date, data_dict, area, target, alignment,
                                train_mod, train_coh, test_mod, test_coh,
                                valid_units, deltas, partial_regress
                            )
                
                gc.collect()
                clear_output(wait=True)
                print(f"✓ Completed: {area}/{target}/mod{train_mod}coh{train_coh}")
    
    print(f"\n{'#'*80}")
    print(f"✓ COMPLETED ALL PROCESSING for {subject} {date}")
    print(f"{'#'*80}")


def process_all_dates(subject, dates, mode='full', 
                     train_conditions='all', test_conditions='all', test_deltas='all',
                     areas=['MST', 'VPS'], targets=['stimulus', 'choice', 'PDW'],
                     permutation_test=False, partial_regress=False):
    """Process all dates with flexible configuration."""
    
    for date in dates:
        print(f"\n{'#'*80}")
        print(f"# PROCESSING DATE: {date}")
        print(f"# MODE: {mode}")
        print(f"{'#'*80}")
        
        try:
            print("Loading session data...")
            data_dict = load_session_data(subject, date)
            
            if mode == 'full':
                run_full_decoding_pipeline(
                    subject, date, data_dict,
                    train_conditions=train_conditions,
                    test_conditions=test_conditions,
                    test_deltas=test_deltas,
                    areas=areas,
                    targets=targets,
                    permutation_test=permutation_test,
                    partial_regress=partial_regress
                )
                
            elif mode == 'train_only':
                train_conds = parse_conditions(train_conditions)
                
                for area in areas:
                    valid_units = data_dict['units_data'][area]
                    for target in targets:
                        for train_mod, train_coh in train_conds:
                            ensure_training_exists(
                                subject, date, data_dict, area, target,
                                train_mod, train_coh, valid_units,
                                permutation_test, partial_regress
                            )
                            gc.collect()
                            
            elif mode == 'delta_only':
                alignments = ['stimOn', 'saccOnset', 'postTargHold']
                train_conds = parse_conditions(train_conditions)
                
                # Parse deltas BEFORE the loop
                print(f"\nParsing delta configuration...")
                deltas = parse_deltas(test_deltas, data_dict['behavior_converted'])
                
                for area in areas:
                    valid_units = data_dict['units_data'][area]
                    for target in targets:
                        for train_mod, train_coh in train_conds:
                            print(f"\n{area}/{target}/mod{train_mod}coh{train_coh}:")
                            for alignment in alignments:
                                test_delta_conditions(
                                    subject, date, data_dict, area, target, alignment,
                                    train_mod, train_coh, valid_units, deltas, partial_regress
                                )
                            gc.collect()
                            
            elif mode == 'cross_only':
                alignments = ['stimOn', 'saccOnset', 'postTargHold']
                train_conds = parse_conditions(train_conditions)
                test_conds = parse_conditions(test_conditions)
                
                # Parse deltas BEFORE the loop
                print(f"\nParsing delta configuration...")
                deltas = parse_deltas(test_deltas, data_dict['behavior_converted'])
                
                for area in areas:
                    valid_units = data_dict['units_data'][area]
                    for target in targets:
                        for train_mod, train_coh in train_conds:
                            for test_mod, test_coh in test_conds:
                                if train_mod == test_mod and train_coh == test_coh:
                                    continue
                                print(f"\n{area}/{target}: train(mod{train_mod}coh{train_coh}) → test(mod{test_mod}coh{test_coh})")
                                for alignment in alignments:
                                    test_cross_condition(
                                        subject, date, data_dict, area, target, alignment,
                                        train_mod, train_coh, test_mod, test_coh,
                                        valid_units, deltas, partial_regress
                                    )
                            gc.collect()
            
            else:
                raise ValueError(f"Unknown mode: {mode}")
                
        except Exception as e:
            print(f"Error processing date {date}: {e}")
            import traceback
            traceback.print_exc()
            continue

DecoderCrossTester type: <class 'type'>
Successfully imported DecoderCrossTester class


### Train the model, and test on same condition

In [ ]:
subject = "zarya"
dates = ["20250602", "20250702", "20250710", "20250523", "20250501", "20250417"]
process_all_dates(subject, dates, mode='train_only', permutation_test=True, partial_regress=True)

### test the delta trial

In [5]:
subject = "zarya"
dates = ["20250602", "20250702", "20250710", "20250523"]
process_all_dates(
    subject=subject,
    dates=dates,
    mode='delta_only',
    train_conditions=[(3, 1), (3, 2)],  
    test_deltas='all',
    areas=['MST', 'VPS'],
    targets=['stimulus', 'choice', 'PDW'],
    partial_regress=False
)


################################################################################
# PROCESSING DATE: 20250602
# MODE: delta_only
################################################################################
Loading session data...
Loaded dots3DMP data: zarya20250602dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250602dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya

Parsing delta configuration...
  Available deltas in data: [-3.0, 0.0, 3.0]
  Using ALL deltas: [-3.0, 0.0, 3.0]

MST/stimulus/mod3coh1:
    → Updating delta tests for stimOn: [-3.0, 0.0, 3.0]
Testing stimulus decoder on NEW condition
Original training: mod=3, coh=1, delta=0
New test: mod=3, coh=1, delta=-3.0
Area: MST, Alignment: stimOn
  Time 0/51
  Time 10/51
  Time 20/51
  Time 30/51
  Time 40/51
  Time 50/51
        ✓ Delta -3.0 tested
Testing stimulus decoder on NEW condition
Original training: mod=3, coh=1, delta=0
New test: mod=3, coh=1, delta=0.0
Area: MST, Alignment: stim

### cross condition testing

In [ ]:
subject = "zarya"
dates = ["20250602", "20250702", "20250710", "20250523", "20250501", "20250417"]


process_all_dates(
    subject=subject,
    dates=dates,
    mode='cross_only',
    train_conditions='all',
    test_conditions='all',
    areas=['MST', 'VPS'],
    targets=['stimulus', 'choice', 'PDW'],
    partial_regress=True
)

